<sub>**The Annotated ML Handbook** · Part VIII: Debugging · by Mahir Absar Khan</sub>

# Chapter 32: How to debug: reading tracebacks and a systematic method

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-8-debugging/32-debug-strategy.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

# How to debug: reading tracebacks and a systematic method

Everyone writes bugs; experienced programmers just find them faster. This chapter teaches a repeatable method: read the error properly, reproduce it small, inspect the state, form a hypothesis, test it. It also covers the tools that make each step quick.

## 32.1 The debugging loop · *Beginner*

> **📖 Definitions**
>
> **Bug**: a mistake in code that makes it fail or give wrong results.
>
> **Debugging**: the systematic process of finding and fixing bugs.
>
> **Debugger**: a tool that pauses a running program so you can inspect variables and step through code line by line, such as `pdb` or the debuggers in VS Code and PyCharm.
>
> **Minimal reproducible example**: the shortest complete piece of code that still shows a problem.


1. **Read** the whole error message, bottom line first.
2. **Locate** the line in *your* code where it happened (the last traceback frame that points to your file).
3. **Inspect** the values involved there: their `type`, `shape`, `dtype`, `len`, a few elements, and whether any are `NaN`.
4. **Hypothesise** one specific cause ("`X` is 1-D because I selected one column with single brackets").
5. **Test** the hypothesis with the smallest possible change, then re-run.
6. **Shrink** the problem if you're still stuck: reproduce it in five lines with fake data. Half the time the bug becomes obvious while you do this; the other half you now have a perfect question for a colleague or a forum.

## 32.2 Anatomy of a traceback · *Beginner*

**A typical failure deep inside a library**

⚠️ *This cell raises an error **on purpose**, so you can see the real message.*

In [2]:
import numpy as np
from sklearn.linear_model import LinearRegression

def fit_model(features, target):
    model = LinearRegression()
    model.fit(features, target)
    return model

heights = np.array([150, 160, 170, 180])
weights = np.array([50, 58, 69, 80])
fit_model(heights, weights)

ValueError: Expected 2D array, got 1D array instead:
array=[150 160 170 180].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import numpy as np</code></td><td>Import NumPy (the array library) under the short name <code>np</code>.</td></tr>
<tr><td><code>from sklearn.linear_model import LinearRegression</code></td><td>Import <code>LinearRegression</code> from scikit-learn&#x27;s <code>linear_model</code> module.</td></tr>
<tr><td><code>def fit_model(features, target):</code></td><td>Define a function <code>fit_model</code> that takes <code>features</code> and <code>target</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;model = LinearRegression()</code></td><td>Create the model.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;model.fit(features, target)</code></td><td>The library call that fails.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return model</code></td><td>Send <code>model</code> back to the caller.</td></tr>
<tr><td><code>heights = np.array([150, 160, 170, 180])</code></td><td>A 1-D array: shape (4,).</td></tr>
<tr><td><code>weights = np.array([50, 58, 69, 80])</code></td><td>Targets.</td></tr>
<tr><td><code>fit_model(heights, weights)</code></td><td>Your top-level call.</td></tr>
</table>
</details>

How to read it:

- **Bottom line** = exception type (`ValueError`) and message. scikit-learn messages often contain the fix ("Reshape your data either using array.reshape(-1, 1)…").
- **Frames** read top-down from your outermost call to where it died. Frames from `<cell>` (or your file names) are *your* code; the "frames inside the library" are scikit-learn's internals. The bug is almost always in the **last frame that is your code**, here `model.fit(features, target)`, because that is where you handed the library something it didn't expect.
- In a full traceback you may see `During handling of the above exception, another exception occurred:` or `The above exception was the direct cause of the following exception:`. Read the **first** exception in the chain for the root cause.

> **💡 Tip** · **Search the message, not your variable names**
>
> Copy the generic part of the error ("Expected 2D array, got 1D array instead") into a search engine or the search box of this book. Remove your own file paths and variable names first.

## 32.3 Inspection tools · *Beginner*

**The five-second state check**

In [3]:
import pandas as pd
def inspect(name, obj):
    """Print the facts that explain 90% of data bugs."""
    info = f"{name}: type={type(obj).__name__}"
    if hasattr(obj, "shape"): info += f" shape={obj.shape}"
    if hasattr(obj, "dtype"): info += f" dtype={obj.dtype}"
    elif hasattr(obj, "dtypes"): info += f" dtypes={dict(obj.dtypes.astype(str))}"
    if hasattr(obj, "isna"): info += f" n_missing={int(obj.isna().sum().sum())}"
    elif isinstance(obj, np.ndarray) and obj.dtype.kind == "f": info += f" n_nan={int(np.isnan(obj).sum())}"
    print(info)

df = pd.DataFrame({"x": [1.0, None, 3.0], "g": ["a", "b", "a"]})
inspect("heights", heights)
inspect("df", df)
inspect("df[['x']]", df[["x"]])
inspect("df['x']", df["x"])

heights: type=ndarray shape=(4,) dtype=int64
df: type=DataFrame shape=(3, 2) dtypes={'x': 'float64', 'g': 'str'} n_missing=1
df[['x']]: type=DataFrame shape=(3, 1) dtypes={'x': 'float64'} n_missing=1
df['x']: type=Series shape=(3,) dtype=float64 n_missing=1


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import pandas as pd</code></td><td>Import pandas (the table library) under the short name <code>pd</code>.</td></tr>
<tr><td><code>def inspect(name, obj):</code></td><td>Define a function <code>inspect</code> that takes <code>name</code> and <code>obj</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;&quot;&quot;Print the facts that explain 90% of data bugs.&quot;&quot;&quot;</code></td><td>Docstring.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;info = f&quot;{name}: type={type(obj).__name__}&quot;</code></td><td>Always show the type.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if hasattr(obj, &quot;shape&quot;): info += f&quot; shape={obj.shape}&quot;</code></td><td>Add the shape if it has one.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if hasattr(obj, &quot;dtype&quot;): info += f&quot; dtype={obj.dtype}&quot;</code></td><td>Arrays and Series have one dtype...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;elif hasattr(obj, &quot;dtypes&quot;): info += f&quot; dtypes={dict(obj.dtypes.astype(str))}&quot;</code></td><td>...DataFrames have one per column.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if hasattr(obj, &quot;isna&quot;): info += f&quot; n_missing={int(obj.isna().sum().sum())}&quot;</code></td><td>pandas objects: count missing values.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;elif isinstance(obj, np.ndarray) and obj.dtype.kind == &quot;f&quot;: info += f&quot; n_nan={int(np.isnan(obj).sum())}&quot;</code></td><td>Float arrays: count NaNs.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(info)</code></td><td>Display the value of <code>info</code>.</td></tr>
<tr><td><code>df = pd.DataFrame({&quot;x&quot;: [1.0, None, 3.0], &quot;g&quot;: [&quot;a&quot;, &quot;b&quot;, &quot;a&quot;]})</code></td><td>A small frame with one missing value.</td></tr>
<tr><td><code>inspect(&quot;heights&quot;, heights)</code></td><td>A 1-D array.</td></tr>
<tr><td><code>inspect(&quot;df&quot;, df)</code></td><td>The whole DataFrame.</td></tr>
<tr><td><code>inspect(&quot;df[[&#x27;x&#x27;]]&quot;, df[[&quot;x&quot;]])</code></td><td>Double brackets: a 2-D DataFrame.</td></tr>
<tr><td><code>inspect(&quot;df[&#x27;x&#x27;]&quot;, df[&quot;x&quot;])</code></td><td>Single brackets: a 1-D Series.</td></tr>
</table>
</details>

### 32.3.1 print debugging, done well · *Beginner*

**f-string debugging**

In [4]:
lr, epoch, loss = 0.01, 3, float("nan")
print(f"{lr=} {epoch=} {loss=}")
assert not np.isnan(loss) or epoch > 0, "loss is NaN at epoch 0"

lr=0.01 epoch=3 loss=nan


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>lr, epoch, loss = 0.01, 3, float(&quot;nan&quot;)</code></td><td>Example values, including a NaN loss.</td></tr>
<tr><td><code>print(f&quot;{lr=} {epoch=} {loss=}&quot;)</code></td><td>The <code>=</code> specifier prints names and values.</td></tr>
<tr><td><code>assert not np.isnan(loss) or epoch &gt; 0, &quot;loss is NaN at epoch 0&quot;</code></td><td>Turn an assumption into a check that fails loudly.</td></tr>
</table>
</details>

### 32.3.2 The interactive debugger · *Intermediate*

`breakpoint()` pauses execution and opens `pdb`, Python's debugger, at that line. You can inspect and change variables, then continue.

**Using breakpoint() and pdb**

*Shown for reference, not run automatically (it waits for input, needs your own files, or would change your system). Copy it into a cell to try it.*

```python
def preprocess(df):
    cleaned = df.dropna()
    breakpoint()
    return cleaned / cleaned.max()

# At the (Pdb) prompt:
#   p cleaned.shape      -> print an expression
#   pp cleaned.dtypes    -> pretty-print
#   l                    -> list the surrounding source code
#   n                    -> execute the next line
#   s                    -> step INTO a function call
#   c                    -> continue running until the next breakpoint
#   u / d                -> move up / down the call stack
#   q                    -> quit
```

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def preprocess(df):</code></td><td>Define a function <code>preprocess</code> that takes <code>df</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;cleaned = df.dropna()</code></td><td>Drop rows with missing values.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;breakpoint()</code></td><td>Execution stops here and a (Pdb) prompt appears.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return cleaned / cleaned.max()</code></td><td>Send <code>cleaned / cleaned.max()</code> back to the caller.</td></tr>
</table>
</details>

> **💡 Tip** · **Post-mortem debugging**
>
> After an exception in Jupyter, run `%debug` in a new cell to open the debugger **at the point of failure**, with every variable intact. In scripts: `python -m pdb my_script.py`. VS Code and PyCharm offer the same with clickable breakpoints and a variable explorer.

### 32.3.3 Warnings are bugs in waiting · *Intermediate*

**Turn warnings into errors while debugging**

In [5]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("error")
    try:
        np.array([1.0, 2.0]) / np.array([0.0, 1.0])
    except RuntimeWarning as w:
        print("caught:", w)

caught: divide by zero encountered in divide


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import warnings</code></td><td>Import the <code>warnings</code> module.</td></tr>
<tr><td><code>with warnings.catch_warnings():</code></td><td>Change warning settings only inside this block.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;warnings.simplefilter(&quot;error&quot;)</code></td><td>Every warning now raises an exception with a full traceback...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;try:</code></td><td>Start a block whose errors we want to catch.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;np.array([1.0, 2.0]) / np.array([0.0, 1.0])</code></td><td>...so a silent divide-by-zero becomes visible.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;except RuntimeWarning as w:</code></td><td>Catch the warning, now raised as an exception.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;print(&quot;caught:&quot;, w)</code></td><td>Show its message.</td></tr>
</table>
</details>

## 32.4 Shrinking the problem · *Beginner*

Build the smallest example that still fails. This one technique solves most hard bugs.

**A minimal reproducible example**

⚠️ *This cell raises an error **on purpose**, so you can see the real message.*

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
X = np.array([[1.0], [2.0], [np.nan]])
y = np.array([0, 1, 0])
LogisticRegression().fit(X, y)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import numpy as np</code></td><td>Imports only what&#x27;s needed.</td></tr>
<tr><td><code>from sklearn.linear_model import LogisticRegression</code></td><td>Import <code>LogisticRegression</code> from scikit-learn&#x27;s <code>linear_model</code> module.</td></tr>
<tr><td><code>X = np.array([[1.0], [2.0], [np.nan]])</code></td><td>Three rows of fake data that trigger the same error.</td></tr>
<tr><td><code>y = np.array([0, 1, 0])</code></td><td>Matching labels.</td></tr>
<tr><td><code>LogisticRegression().fit(X, y)</code></td><td>The one failing call.</td></tr>
</table>
</details>

A good minimal example has fake or tiny data, includes every import, shows the full error, and lists library versions (`pip show scikit-learn`). It's what people on forums and issue trackers need to help you.

## 32.5 Defensive coding for data science · *Intermediate*

**Validate data at the boundaries**

In [7]:
def validate(df: pd.DataFrame) -> pd.DataFrame:
    expected = {"age": "int64", "bmi": "float64", "smoker": "str"}
    missing_cols = set(expected) - set(df.columns)
    if missing_cols:
        raise KeyError(f"missing columns: {sorted(missing_cols)}")
    for col, dt in expected.items():
        if str(df[col].dtype) != dt:
            raise TypeError(f"{col}: expected {dt}, got {df[col].dtype}")
    if not df["age"].between(0, 120).all():
        raise ValueError(f"age out of range: {df.loc[~df['age'].between(0, 120), 'age'].tolist()[:5]}")
    return df

ok = validate(pd.read_csv("insurance.csv"))
print("validated", ok.shape)

validated

 (1000, 7)


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def validate(df: pd.DataFrame) -&gt; pd.DataFrame:</code></td><td>Takes and returns a DataFrame (returns it unchanged if valid).</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;expected = {&quot;age&quot;: &quot;int64&quot;, &quot;bmi&quot;: &quot;float64&quot;, &quot;smoker&quot;: &quot;str&quot;}</code></td><td>The schema we expect: column -&gt; dtype.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;missing_cols = set(expected) - set(df.columns)</code></td><td>Expected columns that are absent.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if missing_cols:</code></td><td>Run the indented block only if <code>missing_cols</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;raise KeyError(f&quot;missing columns: {sorted(missing_cols)}&quot;)</code></td><td>Fail with the list of missing columns.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;for col, dt in expected.items():</code></td><td>Loop over <code>expected.items()</code>, naming each item <code>col, dt</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;if str(df[col].dtype) != dt:</code></td><td>Run the indented block only if <code>str(df[col].dtype) != dt</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;raise TypeError(f&quot;{col}: expected {dt}, got {df[col].dtype}&quot;)</code></td><td>Fail with the offending column and both dtypes.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if not df[&quot;age&quot;].between(0, 120).all():</code></td><td>Run the indented block only if <code>not df[&quot;age&quot;].between(0, 120).all()</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;raise ValueError(f&quot;age out of range: {df.loc[~df[&#x27;age&#x27;].between(0, 120), &#x27;age&#x27;].tolist()[:5]}&quot;)</code></td><td>Fail and show up to 5 bad values.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return df</code></td><td>Send <code>df</code> back to the caller.</td></tr>
<tr><td><code>ok = validate(pd.read_csv(&quot;insurance.csv&quot;))</code></td><td>Validate right after loading.</td></tr>
<tr><td><code>print(&quot;validated&quot;, ok.shape)</code></td><td>Reached only if every check passed.</td></tr>
</table>
</details>

> **🔬 Research practice** · **Tests for analysis code**
>
> Short `pytest` tests (`def test_scaler_has_zero_mean(): ...`) catch regressions when you refactor. Test the parts that are easy to get subtly wrong: data cleaning rules, custom metrics, splitting logic, and any custom transformer.

> **✍️ Try it yourself**
>
> 1. Deliberately cause a `KeyError` in a pandas `groupby`, then use `%debug` (or `python -m pdb`) to inspect the DataFrame's columns at the failure point.
> 2. Write a minimal reproducible example for the last error you were stuck on.
> 3. Add a `validate()` function to your own project's loading code.
>
> *Worked solutions:* [solutions-parts-5-8.ipynb](../../solutions/solutions-parts-5-8.ipynb), section *Chapter 32*.

---

⬅️ [Chapter 31: Reproducibility, leakage checklists and writing it up](../part-7-ml-for-research/31-research-reporting.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 33: The error encyclopedia](33-debug-encyclopedia.ipynb) ➡️